In [7]:
import duckdb

In [ ]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [18]:
df = con.execute("""
                SELECT * FROM (
                    SELECT * , ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row 
                    FROM bronze_z0019
                     WHERE data_ingestao >= '2026-01-06'
                 )
                 WHERE row = 1
                 """).fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10002,PORCA,BT50,200,150,z0019_1.csv,2026-01-06 19:10:19.941106,1
1,10004,SERRA,BT50,100,200,z0019_2.csv,2026-01-06 19:16:21.938083,1
2,10005,MACHADO,BT50,100,60,z0019_2.csv,2026-01-06 19:16:21.938083,1
3,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-01-06 19:10:19.941106,1
4,10003,ARRUELA,BT20,300,250,z0019_2.csv,2026-01-06 19:16:21.938083,1


In [ ]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={"NATBR": "id"})
df_final = df_final.rename(columns={"MAKTX": "nm_produto"})
df_final = df_final.rename(columns={"WERKS": "id_categoria"})
df_final = df_final.rename(columns={"MAINS": "id_fornecedor"})
df_final = df_final.rename(columns={"LABST": "vl_preco"})
df_final.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10002,PORCA,BT50,200,150
1,10004,SERRA,BT50,100,200
2,10005,MACHADO,BT50,100,60
3,10001,PARAFUSO,BT10,100,100
4,10003,ARRUELA,BT20,300,250


In [30]:
df_final.dtypes

id               object
nm_produto       object
id_categoria     object
id_fornecedor    object
vl_preco         object
dtype: object

In [33]:
df2 = df_final
df2 = df2.astype(
    {
        "id": int,
        "nm_produto": str,
        "id_categoria": str,
        "id_fornecedor": int,
        "vl_preco": float,
    }
)
df2.dtypes

id                 int64
nm_produto        object
id_categoria      object
id_fornecedor      int64
vl_preco         float64
dtype: object

In [ ]:
con.execute("""
    CREATE TABLE IF NOT EXISTS produtos (
            id BIGINT
            nm_produto TEXT,
            id_categoria TEXT,
            id_fornecedor BIGINT,
            vl_preco FLOAT
            )
""")

In [ ]:
df2.head(10)
